# PTCG AI Battle — Public Research Bootstrap (FIXED)

**Purpose:** build a research inventory from the attached Kaggle Dataset and, when Internet is enabled, clone the public agent repositories listed in the manifest.

> This notebook does **not** train or run third-party agents. It only downloads/clones source material and inventories files.

In [ ]:
from pathlib import Path
import csv, hashlib, json, os, subprocess, sys, time

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/ptcg_research_index")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

print("Top-level Kaggle inputs:")
for p in INPUT_ROOT.iterdir():
    print("  ", p)


## 1. Find the research dataset

The previous notebook looked only one directory deep. Kaggle can mount a Dataset through an intermediate folder such as `/kaggle/input/datasets/...`, so we search recursively now.

In [ ]:
# Recursively locate the folder that contains our research manifest.
manifest_hits = list(INPUT_ROOT.rglob("public_research_sources.csv"))
if not manifest_hits:
    raise FileNotFoundError(
        "Could not find public_research_sources.csv under /kaggle/input. "
        "Check that the 'PTCG AI Battle Public Research' Dataset is attached."
    )

research_input = manifest_hits[0].parent
print("FOUND research dataset root:")
print(research_input)
print("\nFiles at/under this root:")
for p in sorted(research_input.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(research_input))


## 2. Check Internet before cloning

The research Dataset is only a **seed/manifest**. It does not itself contain every GitHub repository. To fetch the public repositories, Internet must be enabled for this notebook.

Kaggle documents the Internet switch in the notebook Settings/Session Options area. If the switch is unavailable, account/phone verification may be required. 


In [ ]:
import urllib.request

def internet_test():
    try:
        with urllib.request.urlopen("https://github.com", timeout=10) as r:
            return True, r.status
    except Exception as e:
        return False, repr(e)

internet_ok, internet_detail = internet_test()
print("Internet available:", internet_ok)
print("Detail:", internet_detail)


## 3. Clone the six public research repositories

Set `ENABLE_ONLINE_CLONE = True` only after the Internet test above reports `True`.

We clone source code into `/kaggle/working`, not into the Dataset itself.

In [ ]:
REPOS = {
    "scio": "https://github.com/SiamRahman29/pokemon-tcg-agent.git",
    "lawliet": "https://github.com/Lawliet-ai/ptcg-ai-battle-agent.git",
    "ptcg_abc": "https://github.com/wmh/ptcg-abc.git",
    "ducdata": "https://github.com/Ducdata1808/PTCG_Agent.git",
    "pckhoa": "https://github.com/pckhoa206/kaggle_challenge_pokemon.git",
    "scha54": "https://github.com/scha54/Kaggle-Pokemon.git",
}

CLONE_ROOT = WORK_ROOT / "cloned_repos"
CLONE_ROOT.mkdir(parents=True, exist_ok=True)

ENABLE_ONLINE_CLONE = True  # <-- leave True for this research notebook when Internet is ON

clone_results = []
if ENABLE_ONLINE_CLONE:
    if not internet_ok:
        raise RuntimeError(
            "Internet is not available. Turn on Kaggle notebook Internet access, restart/re-run the session, "
            "then rerun this cell."
        )
    for name, url in REPOS.items():
        dest = CLONE_ROOT / name
        if dest.exists():
            print(f"[SKIP] {name}: already exists at {dest}")
            clone_results.append((name, "exists", str(dest)))
            continue
        print(f"[CLONE] {name} <- {url}")
        proc = subprocess.run(
            ["git", "clone", "--depth", "1", url, str(dest)],
            text=True,
            capture_output=True
        )
        if proc.returncode == 0:
            print(f"[OK] {name}")
            clone_results.append((name, "ok", str(dest)))
        else:
            print(f"[FAIL] {name}")
            print(proc.stderr[-2000:])
            clone_results.append((name, "failed", proc.stderr[-2000:]))
else:
    print("Online cloning disabled.")

print("\nClone summary:")
for row in clone_results:
    print(row)


## 4. Inventory notebooks, models, decks and source files

This cell scans the cloned repositories plus any `repos/` snapshots that may already be included in the Dataset.

In [ ]:
roots = []

# Prepared snapshots, if a future Dataset version contains them.
snapshot_root = research_input / "repos"
if snapshot_root.exists():
    roots.append(snapshot_root)

# Public repos cloned above.
if CLONE_ROOT.exists():
    roots.append(CLONE_ROOT)

print("Inventory roots:")
for r in roots:
    print(" ", r)

MODEL_EXT = {
    ".pt", ".pth", ".onnx", ".npz", ".ckpt", ".safetensors",
    ".bin", ".pkl", ".pickle", ".joblib", ".h5", ".keras"
}
NOTEBOOK_EXT = {".ipynb"}
DECK_NAMES = {"deck.csv", "decklist.csv", "deck_list.csv"}

rows = []
for root in roots:
    for f in root.rglob("*"):
        if not f.is_file() or ".git" in f.parts:
            continue
        ext = f.suffix.lower()
        lower_name = f.name.lower()
        if ext in MODEL_EXT:
            kind = "model"
        elif ext in NOTEBOOK_EXT:
            kind = "notebook"
        elif lower_name in DECK_NAMES or "deck" in lower_name and ext in {".csv", ".txt"}:
            kind = "deck"
        elif ext in {".py", ".cpp", ".h", ".hpp", ".md", ".yaml", ".yml", ".json", ".sh"}:
            kind = "source_or_doc"
        else:
            kind = "other"
        rows.append({
            "repo_root": str(root),
            "path": str(f),
            "kind": kind,
            "bytes": f.stat().st_size
        })

artifact_df_path = WORK_ROOT / "research_artifacts.csv"
with artifact_df_path.open("w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=["repo_root","path","kind","bytes"])
    w.writeheader()
    w.writerows(rows)

print("\nTotal files indexed:", len(rows))
from collections import Counter
counts = Counter(r["kind"] for r in rows)
print("By type:", dict(counts))


## 5. Show the important files first

This gives us the files we actually care about for the next research phase.

In [ ]:
import pandas as pd

artifact_df = pd.DataFrame(rows)
if artifact_df.empty:
    print("No repository files were indexed.")
else:
    important = artifact_df[artifact_df["kind"].isin(["model","notebook","deck","source_or_doc"])].copy()
    important = important.sort_values(["kind","bytes"], ascending=[True,False])
    display(important.head(200))


## 6. Produce a summary

This summary is our checkpoint. A successful research bootstrap should normally show nonzero source/notebook counts after the repositories are cloned. Model/deck counts depend on what each public repository actually contains.

In [ ]:
summary = {
    "research_input": str(research_input),
    "internet_available": bool(internet_ok),
    "cloned_repositories": {name: status for name, status, _ in clone_results},
    "artifact_count": int(len(artifact_df)),
    "models": int((artifact_df["kind"] == "model").sum()) if not artifact_df.empty else 0,
    "notebooks": int((artifact_df["kind"] == "notebook").sum()) if not artifact_df.empty else 0,
    "decks": int((artifact_df["kind"] == "deck").sum()) if not artifact_df.empty else 0,
    "source_or_doc": int((artifact_df["kind"] == "source_or_doc").sum()) if not artifact_df.empty else 0,
}

(WORK_ROOT / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))


## 7. Important note about trained models

A Git clone does **not** guarantee that every trained model used in a published solution is in the Git repository. Some competition models were published as Kaggle Notebook outputs or external artifacts instead.

So after this inventory works, our next step is to build a separate **model-artifact acquisition list** for:
- Kaggle notebook outputs/checkpoints
- GitHub-hosted model weights
- ONNX/NPZ/PT/PTH files
- decklists
- replay/data files

Do not assume that `models = 0` means the public research has no trained models.